# Decision Trees: Pruning and Hyperparameter Control
Accompanies **Unit 5.1: Decision Trees from Scratch**.

Decision Trees construct axis-aligned partitions by greedily choosing the split $(j, s)$ that maximizes Gini impurity reduction:
$$\Delta I_G = I_G(S) - \left( \frac{|S_L|}{|S|} I_G(S_L) + \frac{|S_R|}{|S|} I_G(S_R) \right)$$

An unconstrained tree grows until every leaf contains a single sample ($I_G = 0$), memorizing random training noise.

Key Hyperparameters to Tune:
- `max_depth`: Hard constraint on vertical tree depth.
- `min_samples_split`: Minimum number of samples required inside a node before considering a split.
- `min_samples_leaf`: Minimum number of samples required to remain in each child leaf.
- `ccp_alpha`: Cost-Complexity Pruning penalty $\alpha$. Prunes subtrees that fail to decrease impurity by at least $\alpha |T|$.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, train_test_split

data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### Tuning Tree Complexity Constraints via Grid Search

In [ ]:
param_grid = {
    'max_depth': [2, 3, 4, 5, 6],
    'min_samples_leaf': [1, 2, 4, 8],
    'ccp_alpha': [0.0, 0.005, 0.01, 0.02]
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid=param_grid, cv=5)
dt_grid.fit(X_train, y_train)

print(f"Optimal Hyperparameters: {dt_grid.best_params_}")
print(f"Tuned Test Accuracy:    {dt_grid.score(X_test, y_test):.4f}")

### Inspecting the Pruned Tree Architecture

In [ ]:
best_dt = dt_grid.best_estimator_

plt.figure(figsize=(10, 6))
plot_tree(best_dt, feature_names=data.feature_names, class_names=data.target_names, filled=True, rounded=True, fontsize=9)
plt.title(f'Optimally Pruned Decision Tree (max_depth={best_dt.max_depth})', fontsize=12, fontweight='bold')
plt.show()

# Extracting Feature Importance (Mean Decrease in Impurity)
print("\nFeature Importances (MDI):")
for name, imp in zip(data.feature_names, best_dt.feature_importances_):
    print(f"  {name:25s}: {imp:.4f}")